# btp-data-audit
Audit a separately licensed training corpus before student training. This does not turn NH-HAZE evaluation images into training data.

Implementation workflow; no completed GPU experiment is implied. Historical notebooks are preserved.

## Source and runtime
Attach the private research source bundle. Internet is needed to provision the isolated runtime; running this cell downloads dependencies. CPU smoke precedes GPU use. No long experiment runs automatically.

In [ ]:
import hashlib
import json
import os
import shutil
import subprocess
import sys
import time
import zipfile
from pathlib import Path

ROOT = Path("/kaggle/working/BTP")
OUTPUT = Path("/kaggle/working") / ("btp-research-" + str(time.time_ns()))
OUTPUT.mkdir()
# Upload artifacts/btp-kaggle.zip as btp-research-source.bundle in a private dataset.
bundles = list(Path("/kaggle/input").rglob("btp-research-source.bundle"))
if not ROOT.exists():
    assert len(bundles) == 1, (
        "Attach one research source bundle; see docs/RESEARCH_WORKFLOWS.md"
    )
    with zipfile.ZipFile(bundles[0]) as archive:
        for info in archive.infolist():
            destination = (ROOT.parent / info.filename).resolve()
            assert destination.is_relative_to(ROOT), "Unsafe source member"
        archive.extractall(ROOT.parent)
snapshot = json.loads((ROOT / "source-manifest.json").read_text())
for name, expected in snapshot["files"].items():
    path = (ROOT / name).resolve()
    assert path.is_relative_to(ROOT) and path.is_file(), name
    assert hashlib.sha256(path.read_bytes()).hexdigest() == expected, name
print("Verified source files:", len(snapshot["files"]))

In [ ]:
uv = shutil.which("uv")
if uv is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "uv"], check=True)
UV = [uv] if uv else [sys.executable, "-m", "uv"]
PYTHON = Path("/tmp/btp-research-env/bin/python")
if not PYTHON.exists():
    subprocess.run(
        UV + ["venv", "--python", "3.10", str(PYTHON.parent.parent)], check=True
    )
subprocess.run(
    UV
    + [
        "pip",
        "install",
        "--python",
        str(PYTHON),
        "torch==2.6.0",
        "torchvision==0.21.0",
        "--index-url",
        "https://download.pytorch.org/whl/cu118",
    ],
    check=True,
)
subprocess.run(
    UV
    + [
        "pip",
        "install",
        "--python",
        str(PYTHON),
        "-r",
        str(ROOT / "configs/kaggle-requirements.txt"),
    ],
    check=True,
)
RUN = UV + ["run", "--no-project", "--python", str(PYTHON), "python"]
ENVIRONMENT = dict(os.environ, PYTHONPATH=str(ROOT / "src"), CUDA_VISIBLE_DEVICES="0")


def run(arguments):
    subprocess.run(RUN + arguments, cwd=ROOT, env=ENVIRONMENT, check=True)


with (OUTPUT / "runtime-packages.txt").open("w") as handle:
    subprocess.run(
        UV + ["pip", "freeze", "--python", str(PYTHON)], stdout=handle, check=True
    )
(OUTPUT / "source-manifest.json").write_text(json.dumps(snapshot, indent=2))

## Audit training manifest
Set the manifest path and provide the frozen evaluation manifest. The audit checks hashes, paths, paired RGB geometry and split leakage. Manual registration and near-duplicate review remain required.

In [ ]:
TRAIN_MANIFEST = None  # Path to your separate training corpus manifest
EVAL_MANIFEST = ROOT / "configs/evaluation/nh_haze_manifest.json"
if TRAIN_MANIFEST is None:
    print("Training disabled: provide a licensed separate corpus and complete audit.")
else:
    script = """import json, sys
from pathlib import Path
from btp.training_data import audit_training_manifest
p=Path(sys.argv[1]); e=json.loads(Path(sys.argv[2]).read_text())
report=audit_training_manifest(json.loads(p.read_text()), p.parent, evaluation_scenes=e['scenes'])
Path(sys.argv[3]).write_text(json.dumps(report, indent=2))
print(json.dumps(report, indent=2))
"""
    run(
        [
            "-c",
            script,
            str(TRAIN_MANIFEST),
            str(EVAL_MANIFEST),
            str(OUTPUT / "audit.json"),
        ]
    )

In [ ]:
archive = shutil.make_archive(str(OUTPUT), "zip", OUTPUT)
print("Download evidence:", archive)